# Patient-Specific Seizure Detection — LORO Evaluation

Fixed-hyperparameter pipeline (no SVM tuning). For each patient:
LORO over seizure records, per-fold operating-point calibration on
out-of-fold training scores, SzCORE event-level scoring.

**Set your hyperparameters once in the CONFIG cell.** AP / ROC-AUC are
reported per fold as leakage-free diagnostics only — they select nothing.

## Imports

In [1]:
pip install timescoring


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install h5py

Note: you may need to restart the kernel to use updated packages.


In [5]:
def damped_class_weight(y, power=1):
    """Inverse-frequency class weights damped by `power`.
    power=1.0 -> sklearn 'balanced'; power=0.0 -> unweighted; 0.5 -> sqrt.
    Renormalised so the sample-weighted mean weight is 1, so C keeps the
    same effective scale across the ablation. Training labels only."""
    classes, counts = np.unique(y, return_counts=True)
    n = len(y)
    inv = n / (len(classes) * counts)      # == 'balanced' weights
    w = inv ** power                        # damp
    w *= n / np.sum(counts * w)             # mean sample weight -> 1
    return {int(c): float(wi) for c, wi in zip(classes, w)}

# --- SVM class weighting: sqrt-damped inverse frequency ---
# power=0 unweighted, 1.0 = 'balanced' (inverse freq), 0.5 = sqrt midpoint


In [6]:
import os
# single-thread BLAS so the outer joblib parallelism over folds is clean
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"

from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import hashlib
from joblib import Parallel, delayed

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import f_classif
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.svm import SVC
from sklearn.model_selection import (LeaveOneGroupOut, GroupKFold,
                                     StratifiedGroupKFold)  # stratified inner CV
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score)

from timescoring import scoring
from timescoring.annotations import Annotation



## CONFIG

Everything you change between runs lives here. The SVM hyperparameters are
**fixed** — set them once; nothing is tuned on test data.

In [ ]:
CLASS_WEIGHT_POWER = 0.5
# --- patients to run ---
# Sharding: set PATIENT_SHARD env var to run a subset across separate sessions.
#   PATIENT_SHARD="chb04"            -> just the heavy patient, its own session
#   PATIENT_SHARD="chb01,chb02,..."  -> a comma-separated list
#   unset / empty                    -> all 24
import os as _os
_ALL = [f"chb{n:02d}" for n in range(1, 25)]   # chb01 .. chb24
_shard = _os.environ.get("PATIENT_SHARD", "").strip()
if _shard:
    PATIENTS = [p.strip() for p in _shard.split(",") if p.strip()]
else:
    PATIENTS = _ALL

# --- paths ---
SEIZURE_DIR     = Path("Features/Seizure")
NON_SEIZURE_DIR = Path("Features/Non_Seizure")
ANNOTATIONS_CSV = "seizure_annotations.csv"
FEATURE_KEY     = "all_feats"      # dataset name inside the .mat files
LABEL_KEY       = "all_labels"
N_FEATURES      = 1530             # used only to detect transposed arrays

# --- model config ---
# C, gamma AND the feature-selection percentile are ALL auto-tuned per LORO
# fold (see C_GRID / GAMMA_GRID / PERC_GRID below). Nothing is set by hand;
# the empty cfg just carries the fold-tuned values.
cfg = {}

# --- SVM class weighting (effective operating-point knob at threshold 0) --

# --- windowing (must match feature extraction) ---
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

# --- FIXED operating point (a-priori; no calibration) --------------------
# threshold = 0.0 -> SVM native boundary ; k = 2 -> ~12.5s sustained detection.
FIXED_THRESHOLD = 0.0
FIXED_K         = 3
K_SWEEP         = [2,3,4]   # event metrics ALSO reported at each k (thr=0)

# Inner CV used for per-fold C/gamma/percentile tuning (tune_cfg), computed
# on records that EXCLUDE the held-out test record -> leakage-free.
# GroupKFold keeps every record contributing to calibration (~5 fits/fold);
# set to None for full inner leave-one-record-out (clean but ~N fits/fold).
#INNER_CAL_SPLITS  = 3   # CAP on inner folds; actual = min(#seizure records, this)

# === CHANGED (auto-tuned C/gamma per fold) ================================

C_GRID        = [0.1,1,10,100,1000]
GAMMA_GRID    = [0.1, 0.01, 0.001, 0.0001]
K_GRID        = [20,40,60,80,100]

# --- Stage-1 (selection) persistence grid + FP/day ceiling ----------
# K_GRID_CAL sweeps the persistence k (consecutive-positive windows
# required to raise an alarm) during two-stage selection; the operating
# threshold and k are chosen on the seizure-file inner-CV OOF, NOT fixed.
K_GRID_CAL = [3]
# =========================================================================

# --- SzCORE event-scoring parameters (Dan et al., 2024) ---
SZCORE_PARAMS = scoring.EventScoring.Parameters(
    toleranceStart=30, toleranceEnd=60, minOverlap=0,
    maxEventDuration=5 * 60, minDurationBetweenEvents=90,
)


# BOTH classes are thinned into ONE combined budget.
#   - Each seizure record contributes a single CONTIGUOUS chunk that spans all
#     its seizures (first label-1 window to last) plus SEIZ_MARGIN_SEC of
#     interictal padding on EACH side (kept in real-time order).
#   - Non-seizure records then fill whatever remains up to MAX_TRAIN_WINDOWS.
# Test records are still scored in FULL (LORO) -- only TRAINING is thinned.
MAX_TRAIN_WINDOWS = 10000     # TOTAL training windows (seizure chunks + non-seiz)
STRAT_BINS    = 50


# Tuning-only subsample: tune_cfg's inner-CV search fits on at most this many
# windows (ALL seizure windows kept; interictal thinned). The FINAL per-fold
# model fits the full pool; test records scored in full. None -> full pool.
TUNE_MAX_WINDOWS = None
SEIZ_MARGIN_SEC   = 300.0     # interictal padding kept each side of seizure span
# Kept defined for reference but no longer the driver (superseded above):
SUBSAMPLE_SEED            = 0          # base seed (combined per record)
# ==========================================================================

# --- parallelism over LORO folds ---
OUTER_N_JOBS = 4

# --- outputs ---
SUMMARY_CSV = "RESULTS_SUMMARY.csv"   # combined summary across patients

SEIZ = pd.read_csv(ANNOTATIONS_CSV)


# === FCQ-mRMR feature selection (aligned to reference `mrmr` package) ===================================
# Redundancy-aware selection: EEG feature banks (band powers x channels x
# derived ratios) are heavily collinear, so pure-relevance top-k (MI) wastes
# the budget on correlated copies of the same signal. mRMR maximises relevance
# while penalising redundancy -> compact, diverse set (better in p >> n).
#   relevance  = ANOVA F-statistic (f_classif)          -- fast, scale-invariant
#   redundancy = mean |Pearson corr| to already-picked features
#   greedy     = argmax  relevance / redundancy  (FCQ quotient variant)
# Deterministic (no RNG) -> also removes a selection-variance source vs MI.
def mrmr_rank(X, y, k, floor=1e-3):
    """Feature indices ordered by greedy FCQ-mRMR (len <= k). Relevance = ANOVA
    F-statistic (f_classif); redundancy = mean |Pearson corr| to already-selected
    features; score = relevance / redundancy (quotient / FCQ form). ALIGNED to the
    reference `mrmr` package (Mazzanti, the FCQ default): candidate pool restricted
    to relevance>0, each pairwise redundancy floored at `floor` (=0.001), and a
    feature whose mean redundancy to the selected set is 1.0 is rejected (score 0).
    Greedy-forward, so order[:m] is the m-feature subset for any m<=k -> compute
    once at max(k) and slice. Matches the package bit-for-bit on well-conditioned
    data; a 1-2 feature residual can appear only in float-tie / exact-duplicate
    edge cases (immaterial)."""
    X = np.asarray(X, float); y = np.asarray(y)
    n, p = X.shape
    with np.errstate(all="ignore"):
        F, _ = f_classif(X, y)
    F = np.nan_to_num(np.asarray(F, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    F = np.maximum(F, 0.0)
    pool = F > 0.0                                      # package: keep only relevance>0
    k = int(min(k, int(pool.sum())))
    if k <= 0:
        return np.array([], dtype=int)
    # z-score (guard zero-variance) so col dot-products == Pearson correlations
    sd = X.std(0); sd = np.where(sd < 1e-12, 1.0, sd)
    Z = (X - X.mean(0)) / sd
    remaining = pool.copy()
    first = int(np.argmax(np.where(pool, F, -np.inf)))  # step 1: most relevant
    selected = [first]; remaining[first] = False
    accum = np.zeros(p)                                 # sum of FLOORED |corr| to picks
    for _ in range(1, k):
        corr = np.abs((Z.T @ Z[:, selected[-1]]) / n)   # |corr| vs last pick
        corr = np.clip(np.nan_to_num(corr, nan=floor), floor, None)   # per-pair floor
        accum += corr
        den = accum / len(selected)                     # mean |corr| to selected
        den = np.where(np.isclose(den, 1.0), np.inf, den)   # reject perfect-redundant
        score = np.where(remaining, F / den, -np.inf)
        nxt = int(np.argmax(score))
        if not np.isfinite(score[nxt]):
            break
        selected.append(nxt); remaining[nxt] = False
    return np.array(selected, dtype=int)


class MRMRSelector(BaseEstimator, TransformerMixin):
    """Top-k FCQ-mRMR selector. Ranking is fit on TRAINING data only, so it is
    leakage-free inside a Pipeline / CV split (drop-in for SelectKBest)."""
    def __init__(self, k=20):
        self.k = k
    def fit(self, X, y=None):
        self.support_idx_ = mrmr_rank(X, y, self.k)
        return self
    def transform(self, X):
        return np.asarray(X)[:, self.support_idx_]
# =========================================================================


def make_pipe(cfg, class_weight):
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", MRMRSelector(k=cfg["k_feat"])),
        ("svm",    SVC(kernel="rbf", C=cfg["C"], gamma=cfg["gamma"],
                       class_weight=class_weight, cache_size=1024)),
    ])

## Load one patient

Returns `seizure_records` and `non_records` dicts: `{record_name: (X, y)}`.

In [8]:
IMAG_TOL = 1e-6   # |imag| above this = invalid GC window, dropped


def _read_feats(h, key):
    """Read a features array, handling MATLAB-complex compound dtype.
    Returns (X_real_oriented_rows=windows, bad_window_mask)."""
    X = h[key][:]
    if X.dtype.names and "imag" in X.dtype.names:
        real, imag = X["real"], X["imag"]
    else:
        real, imag = X, np.zeros_like(X)
    if real.shape[0] == N_FEATURES:          # orient rows = windows
        real, imag = real.T, imag.T
    bad = np.abs(imag).max(axis=1) > IMAG_TOL
    return np.ascontiguousarray(real, dtype=np.float64), bad


def load_patient(patient):
    records = {}

    for f in sorted(SEIZURE_DIR.glob(f"{patient}*_seizure.mat")):
        try:
            with h5py.File(f, "r") as h:
                X, bad = _read_feats(h, FEATURE_KEY)
                yv = h[LABEL_KEY][:].squeeze()
        except OSError as e:
            print(f"  [SKIP corrupt] {f.name} — {e}")
            continue
        if yv.shape[0] == X.shape[0]:
            y = yv.astype(int)
        else:
            y = yv[: X.shape[0]].astype(int)
        if bad.any():
            print(f"  [DROP complex] {f.name}: {int(bad.sum())} window(s) "
                  f"{list(np.where(bad)[0])}")
            X, y = X[~bad], y[~bad]
        records[f.stem.replace("_seizure", "")] = (X, y)

    for f in sorted(NON_SEIZURE_DIR.glob(f"{patient}*_features.mat")):
        try:
            with h5py.File(f, "r") as h:
                X, bad = _read_feats(h, FEATURE_KEY)
        except OSError as e:
            print(f"  [SKIP corrupt] {f.name} — {e}")
            continue
        if bad.any():
            print(f"  [DROP complex] {f.name}: {int(bad.sum())} window(s) "
                  f"{list(np.where(bad)[0])}")
            X = X[~bad]
        y = np.zeros(X.shape[0], dtype=int)
        records[f.stem.replace("_features", "")] = (X, y)

    seizure_records = {k: v for k, v in records.items() if v[1].sum() > 0}
    non_records     = {k: v for k, v in records.items() if v[1].sum() == 0}
    return seizure_records, non_records

## Event-scoring & calibration helpers

In [9]:
def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN


def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1: i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n


def ref_mask_from_record(record, n, offset_sec=0.0):
    # === CHANGED (seizure-chunk alignment): offset_sec shifts the ABSOLUTE
    # annotation times into the LOCAL coordinates of a subsampled seizure chunk
    # whose window 0 sits offset_sec into the file. Default 0.0 -> identical to
    # the old behavior for full records (test scoring, non-seizure records, and
    # chunks that were clipped to the file start).
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round((r["start_sec"] - offset_sec) * GRID_FS))
        e = int(round((r["end_sec"]   - offset_sec) * GRID_FS))
        if e <= 0 or s >= n:
            continue
        mask[max(s, 0):min(e, n)] = True
    return mask


def _arming_times(sc, thr, k):
    pos = (sc >= thr).astype(int)
    arm, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            arm.append(i * STEP + WINDOW_LEN)
    return arm


def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    arm_times = _arming_times(sc, thr, k)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in arm_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat


def _safe_div(a, b):
    return a / b if b else float("nan")


def segment_metrics(y, sc, thr):
    y = np.asarray(y).astype(int)
    pred = (sc >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    acc  = _safe_div(tp + tn, tp + tn + fp + fn)
    spec = _safe_div(tn, tn + fp)
    sens = _safe_div(tp, tp + fn)
    prec = _safe_div(tp, tp + fp)
    f1 = (_safe_div(2 * prec * sens, prec + sens)
          if (prec == prec and sens == sens) else float("nan"))
    return acc, spec, sens, prec, f1, tp, fp, fn


def event_detections(record, sc, thr, k):
    """One row per ground-truth seizure EVENT in `record`: whether the
    operating point (thr, k) caught it and, if so, the onset latency.
    Uses the SAME arming logic / tolerance window as detection_latencies,
    so the flag and latency stay consistent with the reported metrics."""
    TOL_PRE, TOL_POST = 30, 60
    arm_times = _arming_times(sc, thr, k)
    rows = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in arm_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        rows.append({
            "event_start": onset,
            "event_end":   offset,
            "detected":    1 if cands else 0,
            "latency_s":   (min(cands) - onset) if cands else float("nan"),
        })
    return rows


def event_metrics_at_k(test_outputs, threshold, k):
    """Event-level TP/FP/FN + metrics over ALL test records at (threshold, k).
    Cheap re-scoring of already-computed scores -> a-priori k-robustness table."""
    TP = FP = FN = 0
    for record, sc, y, dur, *_ in test_outputs:
        hyp, n = windows_to_mask(sc, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    total_days = sum(t[3] for t in test_outputs) / 86400
    sens = TP/(TP+FN) if (TP+FN) else 0.0
    prec = TP/(TP+FP) if (TP+FP) else 0.0
    f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
    fpday = FP/total_days if total_days else float("nan")
    return dict(k=k, TP=TP, FP=FP, FN=FN, sensitivity=sens,
                precision=prec, f1=f1, fp_per_day=fpday)


## LORO for one patient

Fixed SVM. Per fold: hold out one record, compute the calibration OOF on the
remaining records (inner grouped CV that EXCLUDES the held-out record), pick
(threshold, k) from those leakage-free scores, fit the deployment model on all
remaining records, and score the held-out record. Seizure records drive
sensitivity (Loop 1); seizure-free records get the same true leave-one-record-out
treatment (Loop 2) so their false alarms count.

`ap` / `roc_auc` are computed per fold on the held-out record as **diagnostics
only** — they are printed, never used to choose anything.

In [ ]:
def _stable_seed(*parts):
    """Deterministic 64-bit seed from arbitrary parts, stable ACROSS processes
    (unlike Python's hash(), which is salted per-process by PYTHONHASHSEED)."""
    s = "|".join("" if p is None else str(p) for p in parts)
    return int.from_bytes(hashlib.blake2b(s.encode(), digest_size=8).digest(),
                          "little")


def _seizure_keep_mask(y, margin_w):
    """Boolean keep-mask over a seizure record's windows: every seizure
    (label==1) window plus +/- margin_w around each seizure RUN. Overlapping
    padded runs merge automatically because they write into one shared mask
    (interval union). Two seizures whose padding touches -> one merged block."""
    keep = np.zeros(len(y), dtype=bool)
    pos = np.where(y == 1)[0]
    if pos.size == 0:
        return keep
    runs = np.split(pos, np.where(np.diff(pos) > 1)[0] + 1)   # contiguous ictal runs
    for run in runs:
        a = max(0, int(run[0]) - margin_w)
        b = min(len(y), int(run[-1]) + 1 + margin_w)
        keep[a:b] = True
    return keep


def _stratified_indices(idx_pool, n_keep, n_bins, rng):
    """Pick n_keep indices spread EVENLY across n_bins contiguous temporal
    segments of the (sorted) idx_pool -> end-to-end coverage of the record
    instead of one clump. Deterministic given rng."""
    idx_pool = np.asarray(idx_pool)
    n = len(idx_pool)
    if n_keep >= n:
        return idx_pool.copy()
    if n_keep <= 0:
        return np.array([], dtype=int)
    bins = np.array_split(idx_pool, min(n_bins, n))
    per, rem = divmod(n_keep, len(bins))
    picks = []
    for bi, b in enumerate(bins):
        q = min(per + (1 if bi < rem else 0), len(b))
        if q > 0:
            picks.append(rng.choice(b, size=q, replace=False))
    return np.sort(np.concatenate(picks)) if picks else np.array([], dtype=int)


def build_train(seizure_records, non_records=None, exclude=None,
                exclude_non=None):
    """Stack TRAINING data (model fits only; test records are scored in full
    elsewhere). Groups are record names (for the inner record-grouped CV).

    Sampling (updated):
      * SEIZURE RECORDS: keep every seizure window (label==1) in full; ALL of the
        record's interictal (label==0) joins the general interictal pool -- no
        peri-ictal margin is kept specially (previously a +/- SEIZ_MARGIN_SEC
        block around each seizure was retained as hard negatives).
      * NON-SEIZURE RECORDS: the whole record joins the general interictal pool.
      * GENERAL POOL: filled up to MAX_TRAIN_WINDOWS - (seizure windows kept),
        allocated per source record proportional to its available interictal
        length and drawn SEGMENT-STRATIFIED (even temporal coverage). Random,
        not contiguous -> maximises interictal diversity. Deterministic per
        (fold, record). ALL seizure windows are always kept.

    Note: the operating point is fixed a-priori (threshold 0, k=3), so training
    windows do NOT need real-time order -> scattered sampling is safe. offsets
    is returned only for call-signature compatibility (unused now)."""
    Xs, ys, grps = [], [], []
    offsets = {}                       # unused (fixed operating point); kept for signature
    # 1) seizure records: keep the seizure windows in full; ALL interictal ->
    #    general pool (undersampled with the non-seizure records). No peri-ictal
    #    margin -- SEIZ_MARGIN_SEC / _seizure_keep_mask no longer used here.
    interictal_pool = []               # list of (name, X, y, avail_idx)
    total_seiz_kept = 0
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        keep = (y == 1)                # CHANGED: seizure windows only (no margin)
        if keep.any():
            Xs.append(X[keep]); ys.append(y[keep])
            grps.extend([name] * int(keep.sum()))
            total_seiz_kept += int(keep.sum())
        rest = np.where(~keep)[0]      # ALL interictal of this record -> pool
        if rest.size:
            interictal_pool.append((name, X, y, rest))
        offsets[name] = 0.0

    # 2) non-seizure records: whole record available to the pool.
    if non_records:
        for name, (X, y) in non_records.items():
            if name == exclude_non:
                continue
            interictal_pool.append((name, X, y, np.arange(len(y))))
            offsets[name] = 0.0

    # 3) fill remaining budget from the pooled interictal, segment-stratified.
    total_avail = sum(len(idx) for _, _, _, idx in interictal_pool)
    budget = (None if MAX_TRAIN_WINDOWS is None
              else max(0, MAX_TRAIN_WINDOWS - total_seiz_kept))
    for name, X, y, idx in interictal_pool:
        if budget is None or total_avail <= budget:
            keep_idx = idx
        elif budget == 0 or total_avail == 0:
            continue
        else:
            quota = max(1, int(round(budget * len(idx) / total_avail)))
            rng = np.random.default_rng(
                _stable_seed(SUBSAMPLE_SEED, exclude, exclude_non, name))
            keep_idx = _stratified_indices(idx, quota, STRAT_BINS, rng)
        if len(keep_idx):
            Xs.append(X[keep_idx]); ys.append(y[keep_idx])
            grps.extend([name] * len(keep_idx))

    return np.vstack(Xs), np.concatenate(ys), np.array(grps), offsets


def _inner_cv(grps, y):
    """Record-grouped, class-STRATIFIED inner CV. StratifiedGroupKFold keeps
    each whole record on one side of every split (no record spans train+val)
    and balances the seizure proportion across folds. The NUMBER of folds is
    driven by the count of SEIZURE records (the binding constraint) -> each fold
    can hold a positive record; capped at INNER_CAL_SPLITS (None -> use all
    seizure records = leave-one-seizure-record-out). With <2 seizure records the
    positive class can't be stratified, so we fall back to plain record-grouped
    KFold and let the pooled OOF F1 tolerate any positive-free fold."""
    grps = np.asarray(grps); y = np.asarray(y)
    n_seiz = len(np.unique(grps[y == 1]))        # seizure-carrying records
    n_tot  = len(np.unique(grps))
    if n_seiz < 2:
        return GroupKFold(n_splits=min(max(2, n_seiz + 1), n_tot))
    k = n_seiz if INNER_CAL_SPLITS is None else min(n_seiz, INNER_CAL_SPLITS)
    return StratifiedGroupKFold(n_splits=max(2, k))


def diag_scores(y, sc):
    y = np.asarray(y).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return float("nan"), float("nan")
    return roc_auc_score(y, sc), average_precision_score(y, sc)


def _subsample_for_tuning(X, y, grps, cap, seed=0):
    """Tuning-only thinning: keep ALL seizure windows, randomly subsample
    interictal so total <= cap. Groups carried in lock-step. Final fit / test
    scoring use full data. (No-op when TUNE_MAX_WINDOWS is None.)"""
    if cap is None or len(y) <= cap:
        return X, y, grps
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    n_neg_keep = max(0, cap - len(pos))
    if n_neg_keep < len(neg):
        neg = np.random.default_rng(seed).choice(neg, size=n_neg_keep, replace=False)
    idx = np.sort(np.concatenate([pos, neg]))
    return X[idx], y[idx], grps[idx]


# def tune_cfg(Xtr, ytr, grps, cfg):
#     """Per-fold selection of (k_feat, C, gamma) by inner record-grouped CV over
#     the fold's training pool (already EXCLUDES the outer held-out record ->
#     leakage-free). Selection metric = POOLED window-F1 at the deployment
#     threshold 0 (beta=1), matching the fixed operating point and the event-F1
#     headline; pooling over all OOF predictions keeps it defined even when an
#     inner fold has no seizure.

#     Efficiency: the MI feature ranking depends only on the fold's training
#     split (not on C/gamma, and k_feat just takes the top-k of one ranking), so
#     mRMR is computed ONCE per inner fold and reused across the whole grid; only
#     the (cheap) SVM is refit per grid point. Returns (winning cfg copy, winning
#     pooled OOF scores)."""
#     if len(np.unique(ytr)) < 2:
#         return {**cfg, "C": 1, "gamma": "scale", "k_feat": 20}, np.zeros(len(ytr))
#     Xt, yt, gt = _subsample_for_tuning(
#         Xtr, ytr, grps, TUNE_MAX_WINDOWS,
#         seed=_stable_seed(SUBSAMPLE_SEED, "tune", len(ytr)))
#     cv = _inner_cv(gt, yt)
#     n_feat = Xt.shape[1]

#     # Precompute per inner fold: scaler, standardised train/val, MI ranking.
#     prep = []
#     for tr, va in cv.split(Xt, yt, groups=gt):
#         scl = StandardScaler().fit(Xt[tr])
#         Xtr_s, Xva_s = scl.transform(Xt[tr]), scl.transform(Xt[va])
#         order = mrmr_rank(Xtr_s, yt[tr], max(K_GRID))  # CHANGED: FCQ-mRMR greedy order; order[:k]==k-feat mRMR set
#         order = mrmr_rank(Xtr_s, yt[tr], max(K_GRID))
#         cw = damped_class_weight(yt[tr], CLASS_WEIGHT_POWER)                       # ADD
#         prep.append((tr, va, Xtr_s, Xva_s, order, cw))         # add cw
#         # prep.append((tr, va, Xtr_s, Xva_s, order)) !!!!!!!

#     best = None   # (f1, C, gamma_value, k_feat, oof_scores)
#     for kf in K_GRID:
#         kf_ = min(kf, n_feat)
#         for C in C_GRID:
#             for gval in GAMMA_GRID:                   # absolute RBF gamma
#                 oof = np.full(len(yt), np.nan)
#                 # for tr, va, Xtr_s, Xva_s, order in prep:
#                 #     sel = order[:kf_]
#                 #     svm = SVC(kernel="rbf", C=C, gamma=gval,
#                 #               class_weight=CLASS_WEIGHT, cache_size=1024)!!!!!
#                 for tr, va, Xtr_s, Xva_s, order, cw in prep:   # add cw
#                     sel = order[:kf_]
#                     svm = SVC(kernel="rbf", C=C, gamma=gval,
#                               class_weight=cw, cache_size=1024) # was CLASS_WEIGHT
#                     svm.fit(Xtr_s[:, sel], yt[tr])
#                     oof[va] = svm.decision_function(Xva_s[:, sel])
#                 m = ~np.isnan(oof)
#                 f1 = f1_score(yt[m], (oof[m] > FIXED_THRESHOLD).astype(int),
#                               zero_division=0)
#                 if best is None or f1 > best[0]:
#                     best = (f1, C, gval, kf_, oof)
#                 # ap = average_precision_score(yt, oof)
#                 # if best is None or ap > best[0]:
#                 #     best = (ap, C, gval, kf_, oof)
#     return {**cfg, "C": best[1], "gamma": best[2], "k_feat": best[3]}, best[4]

# ============================ TWO-STAGE SELECT/DEPLOY ============================
# Stage 1 (SELECTION) and Stage 2 (DEPLOYMENT) are now separated.
#   * Selection picks (C, gamma, k_feat) AND the persistence k + operating
#     threshold on the FULL seizure-containing files only (every window, no
#     undersampling, native real-time order), via inner leave-one-record-out CV.
#   * Deployment refits on the existing UNDERSAMPLED build_train pool (all files,
#     interictal thinned) using the already-chosen (cfg, k, thr) -- no re-tuning.
# tune_cfg is no longer called by the fold processors (kept above for reference).

def build_selection_pool(seizure_records, exclude=None):
    """SELECTION pool: stack EVERY window of the full seizure-containing records
    (NO undersampling), in native real-time order, grouped by record name.
    `exclude` drops the outer fold's test record so selection stays leakage-free.
    Returns (X, y, groups, record_order) or None if empty. Records are stacked in
    order and never shuffled, so each record's rows stay contiguous in real-time
    order -> valid for event-level (windows->alarm) scoring."""
    Xs, ys, grps, order = [], [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(np.asarray(y)); grps.extend([name] * len(y))
        order.append(name)
    if not Xs:
        return None
    return np.vstack(Xs), np.concatenate(ys), np.array(grps), order


def _event_counts_full(record, sc, thr, k, dur):
    """Event-level (SzCORE) TP / FP / FN for ONE record scored in full contiguous
    order at operating point (thr, k). Same masks/params as event_metrics_at_k."""
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS),
                             Annotation(hyp, GRID_FS), SZCORE_PARAMS)
    return int(s.tp), int(s.fp), int(s.refTrue - s.tp)


def select_hp_and_k(seizure_records, cfg, exclude=None):
    """STAGE 1 (selection). On the FULL seizure-file pool (no undersampling,
    native order) with inner LEAVE-ONE-RECORD-OUT CV:
      1. pick (C, gamma, k_feat) by POOLED window-level F1;
      2. pick the operating THRESHOLD by pooled window-level F1 on the winning-hp
         OOF (data-driven, replaces the old FIXED_THRESHOLD=0);
      3. pick the persistence k over K_GRID_CAL, event-level, each held-out record
         scored in full contiguous order: among k with pooled FP/day <=
         FP_PER_DAY_TARGET take MAX event-sensitivity (smallest k breaks ties);
         if none qualify, fall back to best pooled event-F1.
    `exclude` drops the outer fold's test record. Returns
    dict(cfg, k, thr, ap, n_records, ksel). NOTE: FP/day here is estimated only on
    interictal windows INSIDE seizure files, so realized outer FP/day can drift."""
    default = {**cfg, "C": 1, "gamma": "scale", "k_feat": 20}
    pool = build_selection_pool(seizure_records, exclude=exclude)
    if pool is None or len(np.unique(pool[1])) < 2:
        return {"cfg": default, "k": FIXED_K, "thr": FIXED_THRESHOLD,
                "ap": float("nan"), "n_records": 0, "ksel": []}
    X, y, grps, rec_order = pool
    n_feat = X.shape[1]

    # Precompute per inner (leave-one-record-out) fold: scaler, standardised
    # train/val slices, mRMR order (train-split only) and damped class weight.
    # Reused across the whole (C, gamma, k_feat) grid -> only the SVM refits.
    prep = []
    for tr, va in LeaveOneGroupOut().split(X, y, groups=grps):
        if len(np.unique(y[tr])) < 2:
            continue
        scl = StandardScaler().fit(X[tr])
        Xtr_s, Xva_s = scl.transform(X[tr]), scl.transform(X[va])
        order = mrmr_rank(Xtr_s, y[tr], max(K_GRID))
        cw = damped_class_weight(y[tr], CLASS_WEIGHT_POWER)
        prep.append((y[tr], va, Xtr_s, Xva_s, order, cw))
    if not prep:
        return {"cfg": default, "k": FIXED_K, "thr": FIXED_THRESHOLD,
                "ap": float("nan"), "n_records": 0, "ksel": []}

    # ---- Stage 1a: (C, gamma, k_feat) by pooled window-level  ----
    best = None   # (ap, C, gamma, k_feat, oof)

    for kf in K_GRID:
        kf_ = min(kf, n_feat)
        for C in C_GRID:
            for gval in GAMMA_GRID:
                oof = np.full(len(y), np.nan)
                for ytr, va, Xtr_s, Xva_s, order, cw in prep:
                    sel = order[:kf_]
                    svm = SVC(kernel="rbf", C=C, gamma=gval,
                              class_weight=cw, cache_size=1024)
                    svm.fit(Xtr_s[:, sel], ytr)
                    oof[va] = svm.decision_function(Xva_s[:, sel])
                m = ~np.isnan(oof)
                if m.sum() == 0 or len(np.unique(y[m])) < 2:
                    continue
                f1 = f1_score(y[m], (oof[m] > FIXED_THRESHOLD).astype(int),
                              zero_division=0)
                if best is None or f1 > best[0]:
                    best = (f1, C, gval, kf_, oof)
    if best is None:
        return {"cfg": default, "k": FIXED_K, "thr": FIXED_THRESHOLD,
                "ap": float("nan"), "n_records": len(rec_order), "ksel": []}
    win_cfg = {**cfg, "C": best[1], "gamma": best[2], "k_feat": best[3]}
    oof = best[4]


    
    # ---- Stage 1b+1c: joint (threshold, k) by event-level F1 ----
    m = ~np.isnan(oof)
    # high_thresh = oof[m]
    # low_thresh, high_thresh_val = np.quantile(high_thresh, [0.01, 0.99])
    # thr_grid = np.unique(np.linspace(low_thresh, high_thresh_val, 100))
    thr_grid = [FIXED_THRESHOLD]     # pin operating threshold at SVM boundary (0)
    

    recs = []
    for name in rec_order:
        idx = np.where(grps == name)[0]
        sc_r = oof[idx]
        if np.isnan(sc_r).any():
            continue
        recs.append((name, sc_r, y[idx], duration_of(y[idx])))

    cand = []
    for thr in thr_grid:
        for k in K_GRID_CAL:
            TP = FP = FN = 0; days = 0.0
            for name, sc_r, y_r, dur in recs:
                tp, fp, fn = _event_counts_full(name, sc_r, thr, k, dur)
                TP += tp; FP += fp; FN += fn; days += dur / 86400
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prc  = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2 * sens * prc / (sens + prc) if (sens + prc) else 0.0
            fpday = FP / days if days else float("inf")
            cand.append(dict(thr=float(thr), k=k, sens=sens, f1=f1,
                             fpday=fpday, TP=TP, FP=FP, FN=FN))

    chosen = max(cand, key=lambda c: (c["f1"], c["sens"], -c["k"])) if cand \
             else {"thr": FIXED_THRESHOLD, "k": FIXED_K}
    return {"cfg": win_cfg, "k": int(chosen["k"]), "thr": float(chosen["thr"]),
            "ap": float(best[0]), "n_records": len(rec_order), "ksel": cand}


def process_seizure_fold(seizure_records, non_records, test_record, cfg, k, thr):
    """STAGE 2 (deploy) -- LOOP 1 fold. Refit on the UNDERSAMPLED build_train pool
    (all files, interictal thinned) EXCLUDING the held-out seizure record, using
    the already-selected (cfg, k, thr). NO re-tuning. Score held-out record in
    full."""
    Xtr, ytr, grps, offsets = build_train(seizure_records, non_records,
                                          exclude=test_record)
    cw = damped_class_weight(ytr,CLASS_WEIGHT_POWER)
    model = make_pipe(cfg, cw).fit(Xtr, ytr)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    roc, ap = diag_scores(yte, sc)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, roc=roc, ap=ap,
                C=cfg["C"], gamma=cfg["gamma"], k_feat=cfg["k_feat"])


def process_nonseizure_fold(seizure_records, non_records, test_record, cfg, k, thr):
    """STAGE 2 (deploy) -- LOOP 2 fold (TRUE LONO). Refit on the UNDERSAMPLED
    build_train pool EXCLUDING the held-out non-seizure record, using the
    all-seizure selection (cfg, k, thr). NO re-tuning. Score in full."""
    Xtr, ytr, grps, offsets = build_train(seizure_records, non_records,
                                          exclude_non=test_record)
    cw = damped_class_weight(ytr,CLASS_WEIGHT_POWER)
    model = make_pipe(cfg, cw).fit(Xtr, ytr)
    Xte, yte = non_records[test_record]
    sc = model.decision_function(Xte)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, roc=float("nan"), ap=float("nan"),
                C=cfg["C"], gamma=cfg["gamma"], k_feat=cfg["k_feat"])


## Run one patient (build tables + return summary row)

In [11]:
def run_patient(patient, cfg):
    seizure_records, non_records = load_patient(patient)
    if not seizure_records:
        print(f"[{patient}] no seizure records — skipping")
        return None, None, None, None
    print(f"[{patient}] seizure={len(seizure_records)} "
          f"non-seizure={len(non_records)}")

    # --- STAGE 1: parameter SELECTION on full seizure files -------------
    # Per seizure fold, selection EXCLUDES that fold's outer test record
    # (leakage-free). One all-seizure selection (exclude=None) drives the
    # non-seizure loop, whose test records are not in the seizure pool.
    _sel_keys = list(seizure_records.keys())
    _sel_vals = Parallel(n_jobs=OUTER_N_JOBS)(
        delayed(select_hp_and_k)(seizure_records, cfg, r) for r in _sel_keys
    )
    sel = dict(zip(_sel_keys, _sel_vals))
    sel_all = select_hp_and_k(seizure_records, cfg, exclude=None)
    for r in _sel_keys:
        s = sel[r]
        print(f"  [SELECT] {r}  C={s['cfg']['C']} g={s['cfg']['gamma']} "
              f"k_feat={s['cfg']['k_feat']}  k={s['k']} thr={s['thr']:.3f} "
              f"AP={s['ap']:.3f}")
    print(f"  [SELECT-ALL] C={sel_all['cfg']['C']} g={sel_all['cfg']['gamma']} "
          f"k_feat={sel_all['cfg']['k_feat']}  k={sel_all['k']} "
          f"thr={sel_all['thr']:.3f} AP={sel_all['ap']:.3f}")

    # --- LOOP 1: LORO over seizure records (sensitivity / latency) ---
    # Each fold computes its OWN calibration OOF on records that EXCLUDE the
    # held-out test record (nested) -> the operating point is leakage-free.
    seiz_results = Parallel(n_jobs=OUTER_N_JOBS)(
        delayed(process_seizure_fold)(
            seizure_records, non_records, r,
            sel[r]["cfg"], sel[r]["k"], sel[r]["thr"])
        for r in seizure_records.keys()
    )

    test_outputs, meta = [], {}
    for res in seiz_results:
        test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                             res["thr"], res["k"]))
        # === CHANGED: store per-fold tuned C/gamma
        meta[res["record"]] = dict(type="seizure", roc=res["roc"], ap=res["ap"],
                                   C=res["C"], gamma=res["gamma"], k_feat=res["k_feat"])
        print(f"  [SEIZ] {res['record']}  thr={res['thr']:.3f} k={res['k']}  "
              f"C={res['C']} g={res['gamma']} p={res['k_feat']}  "          # CHANGED: tuned hp
              f"roc={res['roc']:.3f} ap={res['ap']:.3f}  "
              f"test_pos={int(res['y'].sum())}")

    # --- LOOP 2: TRUE LONO over non-seizure records (specificity / FP) ---
    if non_records:
        non_results = Parallel(n_jobs=OUTER_N_JOBS)(
            delayed(process_nonseizure_fold)(
                seizure_records, non_records, r,
                sel_all["cfg"], sel_all["k"], sel_all["thr"])
            for r in non_records.keys()
        )
        for res in non_results:
            test_outputs.append((res["record"], res["sc"], res["y"],
                                 res["dur"], res["thr"], res["k"]))
            # === CHANGED: store per-fold tuned C/gamma
            meta[res["record"]] = dict(type="nonseizure", roc=float("nan"),
                                       ap=float("nan"),
                                       C=res["C"], gamma=res["gamma"], k_feat=res["k_feat"])
        print(f"  [NON] {len(non_results)} records scored (true LONO)")

    # --- per-record pass: segment table + event totals ---
    TP = FP = FN = 0
    sTP = sFP = sFN = 0   # 1 Hz sample-based accumulators (SzCORE 2.3)
    all_latencies, segment_rows, event_rows, fp_rows = [], [], [], []
    for record, sc, y, dur, thr, k in test_outputs:
        is_seiz = meta[record]["type"] == "seizure"
        hyp, n = windows_to_mask(sc, dur, thr, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
        # === 1 Hz sample-based scoring on the SAME masks (no tolerances/merge)
        ss = scoring.SampleScoring(Annotation(ref, GRID_FS),
                                   Annotation(hyp, GRID_FS))
        sTP += ss.tp; sFP += ss.fp; sFN += ss.refTrue - ss.tp
        # === false-positive table: one row per hypothesis (alarm) event that
        # SzCORE counts as a false detection. Mirrors the library's own FP
        # loop (hyp events not overlapping tpMask) so counts match s.fp.
        for _ev in s.hyp.events:
            _a, _b = round(_ev[0] * s.fs), round(_ev[1] * s.fs)
            if np.all(~s.tpMask[_a:_b]):
                fp_rows.append({
                    "patient": patient, "record": record,
                    "fp_start": _ev[0], "fp_duration_s": _ev[1] - _ev[0],
                })
        if is_seiz:
            all_latencies.extend(detection_latencies(record, sc, thr, k))
            # === event-level table: one row per ground-truth seizure event
            for ev in event_detections(record, sc, thr, k):
                event_rows.append({
                    "patient": patient, "record": record,
                    "event_start": ev["event_start"],
                    "event_end":   ev["event_end"],
                    "detected":    ev["detected"],
                    "latency_s":   ev["latency_s"],
                    "threshold": thr, "k": k,
                    "k_feat": meta[record]["k_feat"],
                })

        acc, spec, sens, prec, f1, stp, sfp, sfn = segment_metrics(y, sc, 0.0)
        if not is_seiz:
            acc = spec = sens = prec = f1 = np.nan
        segment_rows.append({
            # === CHANGED: 'config' column dropped (constant label, no info)
            "C": meta[record]["C"],
            "gamma": meta[record]["gamma"],
            "k_feat": meta[record]["k_feat"],
            "patient": patient, "record": record,
            "n_windows": int(len(y)), "n_seizure_windows": int(np.sum(y)),
            "record_length": dur, "threshold": thr, "k": k,
            "roc_auc": meta[record]["roc"], "ap": meta[record]["ap"],
            "seg_accuracy": acc, "seg_specificity": spec,
            "seg_sensitivity": sens, "seg_precision": prec, "seg_f1": f1,
            "tp": stp, "fp": sfp, "fn": sfn,
        })
        kind = "SEIZ" if is_seiz else "NON"
        print(f"    {record} done  ({kind})", flush=True)

    # --- event-level totals ---
    sens = TP/(TP+FN) if (TP+FN) else 0.0
    prec = TP/(TP+FP) if (TP+FP) else 0.0
    f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
    # === 1 Hz sample-based metrics (SzCORE 2.3; complementary to event-based)
    s_sens = sTP/(sTP+sFN) if (sTP+sFN) else 0.0
    s_prec = sTP/(sTP+sFP) if (sTP+sFP) else 0.0
    s_f1   = 2*s_sens*s_prec/(s_sens+s_prec) if (s_sens+s_prec) else 0.0
    total_len_sec = sum(t[3] for t in test_outputs)
    days = total_len_sec / 86400
    med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")

    fp_day = FP / days if days else float("nan")
    print(f"  EVENT: TP={TP} FP={FP} FN={FN}  sens={sens:.3f} "
          f"prec={prec:.3f} f1={f1:.3f}  FP/day={fp_day:.3f}  "
          f"lat={med_lat:.1f}s")
    print(f"  SAMPLE(1Hz): TP={sTP} FP={sFP} FN={sFN}  "
          f"sens={s_sens:.3f} prec={s_prec:.3f} f1={s_f1:.3f}")

    # --- a-priori k-robustness sweep (threshold fixed at 0; reporting only) ---
    ksweep = [event_metrics_at_k(test_outputs, FIXED_THRESHOLD, kk) for kk in K_SWEEP]
    print("  K-SWEEP (thr=0):")
    for _m in ksweep:
        print(f"    k={_m['k']}: sens={_m['sensitivity']:.3f} "
              f"prec={_m['precision']:.3f} f1={_m['f1']:.3f} "
              f"FP/day={_m['fp_per_day']:.3f}  (TP={_m['TP']} FP={_m['FP']} FN={_m['FN']})")

    # === CHANGED: C/gamma now vary per fold -> report the MODAL choice for the
    # patient (per-fold values are in the segment CSV).
    _cs = [m.get("C") for m in meta.values() if m.get("C") is not None]
    _gs = [m.get("gamma") for m in meta.values() if m.get("gamma") is not None]
    mode_C     = max(set(_cs), key=_cs.count) if _cs else None
    mode_gamma = max(set(_gs), key=_gs.count) if _gs else None
    _ps = [m.get("k_feat") for m in meta.values() if m.get("k_feat") is not None]
    mode_perc  = max(set(_ps), key=_ps.count) if _ps else None

    summary_row = {
        # === CHANGED: 'config' column dropped (constant label, no info)
        "C": mode_C, "gamma": mode_gamma,
        "k_feat": mode_perc,
        "patient": patient,
        "total_length_hours": total_len_sec / 3600,
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "precision": prec, "f1": f1,
        "sample_TP": sTP, "sample_FP": sFP, "sample_FN": sFN,
        "sample_sensitivity": s_sens, "sample_precision": s_prec, "sample_f1": s_f1,
        "FP/day": FP / days if days else float("nan"),
        "FP/hour": FP / (days * 24) if days else float("nan"),
        "latency_median": med_lat, "n_detected": len(all_latencies),
        "mean_fold_roc_auc": float(np.nanmean(
            [m["roc"] for m in meta.values()])),
        "mean_fold_ap": float(np.nanmean([m["ap"] for m in meta.values()])),
    }
    for _m in ksweep:
        _k=_m["k"]
        summary_row[f"sens_k{_k}"]=_m["sensitivity"]; summary_row[f"f1_k{_k}"]=_m["f1"]
        summary_row[f"fpday_k{_k}"]=_m["fp_per_day"]

    return summary_row, segment_rows, event_rows, fp_rows


# Run

Auto-tuned C/gamma per fold. Runs every patient in `PATIENTS` (set the `PATIENT_SHARD` env var to subset).

## chb01

In [ ]:
PATIENT = "chb01"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb02"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb03"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb04"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb05"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


## chb06

In [12]:
PATIENT = "chb06"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb06
[chb06] seizure=7 non-seizure=11


KeyboardInterrupt: 

In [ ]:
PATIENT = "chb07"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb08"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb09"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb10"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


## chb11

In [ ]:
PATIENT = "chb11"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb12"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb13"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb14"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb15"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


## chb16

In [ ]:
PATIENT = "chb16"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb17"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb18"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb19"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb20"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


## chb21

In [87]:
PATIENT = "chb21"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb01
[chb01] seizure=7 non-seizure=35


KeyboardInterrupt: 

In [ ]:
PATIENT = "chb22"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb11
[chb11] seizure=3 non-seizure=32


KeyboardInterrupt: 

In [ ]:
PATIENT = "chb23"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb13


KeyboardInterrupt: 

In [ ]:
PATIENT = "chb24"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb02
[chb02] seizure=3 non-seizure=33
  [SELECT] chb02_16+  C=1000 g=0.01 k_feat=20  k=6 thr=-1.874 AP=0.038
  [SELECT] chb02_16  C=1000 g=0.001 k_feat=100  k=6 thr=-0.034 AP=0.021
  [SELECT] chb02_19  C=1000 g=0.001 k_feat=100  k=6 thr=-0.043 AP=0.158
  [SELECT-ALL] C=1000 g=0.001 k_feat=100  k=6 thr=-0.883 AP=0.058
  [SEIZ] chb02_16+  thr=-1.874 k=6  C=1000 g=0.01 p=20  roc=0.692 ap=0.079  test_pos=33
  [SEIZ] chb02_16  thr=-0.034 k=6  C=1000 g=0.001 p=100  roc=0.757 ap=0.444  test_pos=33
  [SEIZ] chb02_19  thr=-0.043 k=6  C=1000 g=0.001 p=100  roc=0.866 ap=0.021  test_pos=4
  [NON] 33 records scored (true LONO)
    chb02_16+ done  (SEIZ)
    chb02_16 done  (SEIZ)
    chb02_19 done  (SEIZ)
    chb02_01 done  (NON)
    chb02_02 done  (NON)
    chb02_03 done  (NON)
    chb02_04 done  (NON)
    chb02_05 done  (NON)
    chb02_06 done  (NON)
    chb02_07 done  (NON)
    chb02_08 done  (NON)
    chb02_09 done  (NON)
    chb02_10 don

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4,sens_k5,f1_k5,fpday_k5
0,1000,0.001,100,chb02,46.265972,0,0,3,0.0,0.0,...,6.743617,0.333333,0.333333,1.03748,0.333333,0.5,0.0,0.0,0.0,0.0


In [95]:
all_segments

[]

## chb02 - weights = 0

In [120]:
CLASS_WEIGHT_POWER=0
PATIENT = "chb02"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb02
[chb02] seizure=3 non-seizure=33
  [SELECT] chb02_16+  C=0.1 g=0.01 k_feat=20  k=2 thr=-1.030 AP=0.039
  [SELECT] chb02_16  C=10 g=0.001 k_feat=100  k=4 thr=-0.062 AP=0.026
  [SELECT] chb02_19  C=1000 g=0.001 k_feat=100  k=2 thr=-0.043 AP=0.158
  [SELECT-ALL] C=1000 g=0.001 k_feat=100  k=4 thr=-0.883 AP=0.058
  [SEIZ] chb02_16+  thr=-1.030 k=2  C=0.1 g=0.01 p=20  roc=0.746 ap=0.065  test_pos=33
  [SEIZ] chb02_16  thr=-0.062 k=4  C=10 g=0.001 p=100  roc=0.738 ap=0.297  test_pos=33
  [SEIZ] chb02_19  thr=-0.043 k=2  C=1000 g=0.001 p=100  roc=0.796 ap=0.019  test_pos=4
  [NON] 33 records scored (true LONO)
    chb02_16+ done  (SEIZ)
    chb02_16 done  (SEIZ)
    chb02_19 done  (SEIZ)
    chb02_01 done  (NON)
    chb02_02 done  (NON)
    chb02_03 done  (NON)
    chb02_04 done  (NON)
    chb02_05 done  (NON)
    chb02_06 done  (NON)
    chb02_07 done  (NON)
    chb02_08 done  (NON)
    chb02_09 done  (NON)
    chb02_10 done  (NO

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4,sens_k5,f1_k5,fpday_k5
0,1000,0.001,100,chb02,46.265972,1,28,2,0.333333,0.034483,...,11.412275,0.0,0.0,2.593699,0.0,0.0,0.51874,0.0,0.0,0.0


In [122]:
pd.DataFrame(all_segments)

,C,gamma,k_feat,patient,record,n_windows,n_seizure_windows,record_length,threshold,k,roc_auc,ap,seg_accuracy,seg_specificity,seg_sensitivity,seg_precision,seg_f1,tp,fp,fn
0,0.1,0.010,20,chb02,chb02_16+,1437,33,3600.0,-1.030431,2,0.746417,0.065462,0.977035,1.000000,0.0,NaN,NaN,0,0,33
1,10.0,0.001,100,chb02,chb02_16,380,33,957.5,-0.061936,4,0.737578,0.297239,0.913158,1.000000,0.0,NaN,NaN,0,0,33
2,1000.0,0.001,100,chb02,chb02_19,1437,4,3600.0,-0.043050,2,0.795534,0.019385,0.983994,0.986741,0.0,0.0,NaN,0,19,4
3,1000.0,0.001,100,chb02,chb02_01,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2,0
4,1000.0,0.001,100,chb02,chb02_02,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
5,1000.0,0.001,100,chb02,chb02_03,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
6,1000.0,0.001,100,chb02,chb02_04,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
7,1000.0,0.001,100,chb02,chb02_05,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
8,1000.0,0.001,100,chb02,chb02_06,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3,0
9,1000.0,0.001,100,chb02,chb02_07,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,6,0


## chb02 - weights=1

In [124]:
CLASS_WEIGHT_POWER=1

PATIENT = "chb02"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb02
[chb02] seizure=3 non-seizure=33
  [SELECT] chb02_16+  C=1000 g=0.01 k_feat=20  k=3 thr=-1.874 AP=0.038
  [SELECT] chb02_16  C=1000 g=0.001 k_feat=100  k=4 thr=-0.034 AP=0.021
  [SELECT] chb02_19  C=1000 g=0.0001 k_feat=100  k=4 thr=1.461 AP=0.163
  [SELECT-ALL] C=1000 g=0.001 k_feat=100  k=4 thr=-0.883 AP=0.058
  [SEIZ] chb02_16+  thr=-1.874 k=3  C=1000 g=0.01 p=20  roc=0.763 ap=0.156  test_pos=33
  [SEIZ] chb02_16  thr=-0.034 k=4  C=1000 g=0.001 p=100  roc=0.825 ap=0.481  test_pos=33
  [SEIZ] chb02_19  thr=1.461 k=4  C=1000 g=0.0001 p=100  roc=0.771 ap=0.017  test_pos=4
  [NON] 33 records scored (true LONO)
    chb02_16+ done  (SEIZ)
    chb02_16 done  (SEIZ)
    chb02_19 done  (SEIZ)
    chb02_01 done  (NON)
    chb02_02 done  (NON)
    chb02_03 done  (NON)
    chb02_04 done  (NON)
    chb02_05 done  (NON)
    chb02_06 done  (NON)
    chb02_07 done  (NON)
    chb02_08 done  (NON)
    chb02_09 done  (NON)
    chb02_10 don

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4,sens_k5,f1_k5,fpday_k5
0,1000,0.001,100,chb02,46.265972,2,16,1,0.666667,0.111111,...,12.968494,1.0,0.428571,4.149918,0.333333,0.333333,1.03748,0.0,0.0,0.0


In [128]:
pd.DataFrame(all_segments)

,C,gamma,k_feat,patient,record,n_windows,n_seizure_windows,record_length,threshold,k,roc_auc,ap,seg_accuracy,seg_specificity,seg_sensitivity,seg_precision,seg_f1,tp,fp,fn
0,1000,0.0100,20,chb02,chb02_16+,1437,33,3600.0,-1.873892,3,0.763295,0.156192,0.970077,0.989316,0.151515,0.250000,0.188679,5,15,28
1,1000,0.0010,100,chb02,chb02_16,380,33,957.5,-0.034401,4,0.824906,0.481421,0.923684,0.979827,0.333333,0.611111,0.431373,11,7,22
2,1000,0.0001,100,chb02,chb02_19,1437,4,3600.0,1.460995,4,0.771458,0.016515,0.939457,0.940684,0.500000,0.022989,0.043956,2,85,2
3,1000,0.0010,100,chb02,chb02_01,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2,0
4,1000,0.0010,100,chb02,chb02_02,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
5,1000,0.0010,100,chb02,chb02_03,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
6,1000,0.0010,100,chb02,chb02_04,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
7,1000,0.0010,100,chb02,chb02_05,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
8,1000,0.0010,100,chb02,chb02_06,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,5,0
9,1000,0.0010,100,chb02,chb02_07,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,6,0


## chb02 - weights = 0.5

In [ ]:
CLASS_WEIGHT_POWER=0.5
PATIENT = "chb02"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb02
[chb02] seizure=3 non-seizure=33
  [SELECT] chb02_16+  C=0.1 g=0.01 k_feat=20  k=2 thr=-1.030 AP=0.039
  [SELECT] chb02_16  C=10 g=0.001 k_feat=100  k=4 thr=-0.062 AP=0.026
  [SELECT] chb02_19  C=1000 g=0.001 k_feat=100  k=2 thr=-0.043 AP=0.158
  [SELECT-ALL] C=1000 g=0.001 k_feat=100  k=4 thr=-0.883 AP=0.058
  [SEIZ] chb02_16+  thr=-1.030 k=2  C=0.1 g=0.01 p=20  roc=0.746 ap=0.065  test_pos=33
  [SEIZ] chb02_16  thr=-0.062 k=4  C=10 g=0.001 p=100  roc=0.738 ap=0.297  test_pos=33
  [SEIZ] chb02_19  thr=-0.043 k=2  C=1000 g=0.001 p=100  roc=0.796 ap=0.019  test_pos=4
  [NON] 33 records scored (true LONO)
    chb02_16+ done  (SEIZ)
    chb02_16 done  (SEIZ)
    chb02_19 done  (SEIZ)
    chb02_01 done  (NON)
    chb02_02 done  (NON)
    chb02_03 done  (NON)
    chb02_04 done  (NON)
    chb02_05 done  (NON)
    chb02_06 done  (NON)
    chb02_07 done  (NON)
    chb02_08 done  (NON)
    chb02_09 done  (NON)
    chb02_10 done  (NO

,C,gamma,k_feat,patient,total_length_hours,TP,FP,FN,sensitivity,precision,...,fpday_k2,sens_k3,f1_k3,fpday_k3,sens_k4,f1_k4,fpday_k4,sens_k5,f1_k5,fpday_k5
0,1000,0.001,100,chb02,46.265972,1,28,2,0.333333,0.034483,...,11.412275,0.0,0.0,2.593699,0.0,0.0,0.51874,0.0,0.0,0.0


In [110]:
pd.DataFrame(all_segments)

,C,gamma,k_feat,patient,record,n_windows,n_seizure_windows,record_length,threshold,k,roc_auc,ap,seg_accuracy,seg_specificity,seg_sensitivity,seg_precision,seg_f1,tp,fp,fn
0,0.1,0.010,20,chb02,chb02_16+,1437,33,3600.0,-1.030431,2,0.746417,0.065462,0.977035,1.000000,0.0,NaN,NaN,0,0,33
1,10.0,0.001,100,chb02,chb02_16,380,33,957.5,-0.061936,4,0.737578,0.297239,0.913158,1.000000,0.0,NaN,NaN,0,0,33
2,1000.0,0.001,100,chb02,chb02_19,1437,4,3600.0,-0.043050,2,0.795534,0.019385,0.983994,0.986741,0.0,0.0,NaN,0,19,4
3,1000.0,0.001,100,chb02,chb02_01,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2,0
4,1000.0,0.001,100,chb02,chb02_02,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
5,1000.0,0.001,100,chb02,chb02_03,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
6,1000.0,0.001,100,chb02,chb02_04,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
7,1000.0,0.001,100,chb02,chb02_05,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
8,1000.0,0.001,100,chb02,chb02_06,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3,0
9,1000.0,0.001,100,chb02,chb02_07,1917,0,4800.0,-0.883488,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,6,0
